In [1]:
#1
!pip install -q tensorflow

#  Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive/CSV_of_the_otherdataset/

ls: cannot access '/content/drive/MyDrive/CSV_of_the_otherdataset/': No such file or directory


In [3]:
#2
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split


np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print(f" TensorFlow version: {tf.__version__}")
print(f" GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

 TensorFlow version: 2.20.0
 GPU available: True


In [ ]:
#3
BASE_PATH = '/content/drive/MyDrive/GP/CSV_completeDataset'

def load_csvs(base_path, split):
    all_dfs = []
    for class_name, label in [('Alert', 0), ('Drowsy', 1)]:
        folder = os.path.join(base_path, split, class_name)
        files = glob.glob(os.path.join(folder, '*.csv'))

        for f in files:
            df = pd.read_csv(f)
            filename   = os.path.basename(f).replace('.csv', '')
            video_id   = filename
            subject_id = filename.split('_')[0]

            df['video_id']   = video_id
            df['subject_id'] = subject_id
            df['label']      = label
            df['class_name'] = class_name
            all_dfs.append(df)

    return pd.concat(all_dfs, ignore_index=True)

train_df = load_csvs(BASE_PATH, 'training')
valid_df = load_csvs(BASE_PATH, 'valid')
test_df  = load_csvs(BASE_PATH, 'testing')

print("=" * 50)

print(f"Training data: ")
print(f"   Number of videos: {train_df['video_id'].nunique()}")
print(f"   Total blinks: {len(train_df)}")

print(f"\nValidation data:")
print(f"   Number of videos: {valid_df['video_id'].nunique()}")
print(f"   Total blinks: {len(valid_df)}")

print(f"\nTest data:")
print(f"   Number of videos: {test_df['video_id'].nunique()}")
print(f"   Total blinks: {len(test_df)}")

In [ ]:
#cross dataset

import os
import glob
import pandas as pd

# Main dataset path
BASE_PATH = '/content/drive/MyDrive/GP/CSV_completeDataset'

# Other dataset path (used only for testing)
OTHER_TEST_PATH = '/content/drive/MyDrive/GP/CSV_of_the_otherdataset'

def load_csvs(base_path, split):
    all_dfs = []

    for class_name, label in [('Alert', 0), ('Drowsy', 1)]:

        folder = os.path.join(base_path, split, class_name)
        files = glob.glob(os.path.join(folder, '*.csv'))

        for f in files:
            df = pd.read_csv(f)

            filename = os.path.basename(f).replace('.csv', '')
            video_id = filename
            subject_id = filename.split('_')[0]

            # Add metadata columns
            df['video_id'] = video_id
            df['subject_id'] = subject_id
            df['label'] = label
            df['class_name'] = class_name

            all_dfs.append(df)

    return pd.concat(all_dfs, ignore_index=True)

# Load training and validation from original dataset
train_df = load_csvs(BASE_PATH, 'training')
valid_df = load_csvs(BASE_PATH, 'valid')

# Load testing from the OTHER dataset
test_df = load_csvs(OTHER_TEST_PATH, 'testing')

# =========================
# Dataset statistics
# =========================

print("=" * 50)

print("Training data:")
print(f"   Number of videos: {train_df['video_id'].nunique()}")
print(f"   Total blinks: {len(train_df)}")

print("\nValidation data:")
print(f"   Number of videos: {valid_df['video_id'].nunique()}")
print(f"   Total blinks: {len(valid_df)}")

print("\nTest data:")
print(f"   Number of videos: {test_df['video_id'].nunique()}")
print(f"   Total blinks: {len(test_df)}")

In [ ]:
#4
#defult if I don't choose a value for them
def build_window_features(df, window_size=8, stride=2):


#Builds windows using only raw values ​​(4 features) Suitable for LSTM

    feature_cols = ['Duration', 'Amplitude', 'Velocity', 'Frequency']
    rows = []

    for video_id, g in df.groupby('video_id'):
        g = g.sort_values('Blink_ID').reset_index(drop=True)

        if len(g) < window_size:
            continue

        label      = g['label'].iloc[0]
        subject_id = g['subject_id'].iloc[0]

        for start in range(0, len(g) - window_size + 1, stride):
            window = g.iloc[start:start + window_size]

            feat = {
                'video_id': video_id,
                'subject_id': subject_id,
                'label': label,
                'window_idx': start,

                'data': window[feature_cols].values
            }

            rows.append(feat)

    return pd.DataFrame(rows)


# ================================
WINDOW_SIZE = 20
STRIDE = 2

print(" Building training windows...")
train_windows = build_window_features(train_df, WINDOW_SIZE, STRIDE)
print(f" Training windows: {len(train_windows)}")

print("\n Building validation windows...")
valid_windows = build_window_features(valid_df, WINDOW_SIZE, STRIDE)
print(f" validation windows: {len(valid_windows)}")

print("\n Building test windows...")
test_windows = build_window_features(test_df, WINDOW_SIZE, STRIDE)
print(f" Test windows: {len(test_windows)}")


feature_cols = ['Duration', 'Amplitude', 'Velocity', 'Frequency']
print(f"\n Number of features per window: {len(feature_cols)}")

In [ ]:
#5
def windows_to_sequences(windows_df, sequence_length=5):
    sequences = []
    labels = []
    video_ids = []
    subject_ids = []

    for video_id, g in windows_df.groupby('video_id'):
        g = g.sort_values('window_idx').reset_index(drop=True)

        if len(g) < sequence_length:
            continue

        for i in range(len(g) - sequence_length + 1):

            window_seq = g.iloc[i:i + sequence_length]['data'].tolist()

            window_seq = [w.reshape(-1) for w in window_seq]

            seq = np.array(window_seq, dtype=np.float32)

            sequences.append(seq)
            labels.append(g['label'].iloc[i])
            video_ids.append(video_id)
            subject_ids.append(g['subject_id'].iloc[i])

    return (np.array(sequences, dtype=np.float32),
            np.array(labels),
            np.array(video_ids),
            np.array(subject_ids))


# =================================
SEQUENCE_LENGTH = 5


print(f"Convert windows to sequences {SEQUENCE_LENGTH}")
print()

X_train_seq, y_train, train_video_ids, _ = windows_to_sequences(
    train_windows, SEQUENCE_LENGTH)

X_valid_seq, y_valid, valid_video_ids, _ = windows_to_sequences(
    valid_windows, SEQUENCE_LENGTH)

X_test_seq, y_test, test_video_ids, _ = windows_to_sequences(
    test_windows, SEQUENCE_LENGTH)


#(Samples, Time Steps, Features)

print(f" Training data shape:  {X_train_seq.shape}")
print(f" Test data shape: {X_test_seq.shape}")

print(f"\n Class distribution in training:")
print(f"   Alert:  {(y_train == 0).sum()}")
print(f"   Drowsy: {(y_train == 1).sum()}")

Convert windows to sequences 5

 Training data shape:  (1358, 5, 80)
 Test data shape: (537, 5, 80)

 Class distribution in training:
   Alert:  576
   Drowsy: 782


In [ ]:

#6
X_train_seq = np.array(X_train_seq.tolist(), dtype=np.float32)
X_valid_seq = np.array(X_valid_seq.tolist(), dtype=np.float32)
X_test_seq  = np.array(X_test_seq.tolist(),  dtype=np.float32)

y_train = np.array(y_train, dtype=np.int32)
y_valid = np.array(y_valid, dtype=np.int32)
y_test  = np.array(y_test,  dtype=np.int32)




print("Data type:")
print("X_train:", X_train_seq.dtype)
print("y_train:", y_train.dtype)

print("\nShape:")
print("Train:", X_train_seq.shape)
print("Valid:", X_valid_seq.shape)
print("Test:", X_test_seq.shape)

Data type:
X_train: float32
y_train: int32

Shape:
Train: (1358, 5, 80)
Valid: (239, 5, 80)
Test: (537, 5, 80)


In [ ]:
#7
#LSTM with two layers :

# - LSTM Layer 1 (96 units) : Understands patterns

# - LSTM Layer 2 (48 units) : Summarizes information

# - Dense : Final classification


def build_lstm_model(sequence_length, n_features):



    model = Sequential([

        LSTM(96,
             return_sequences=True,
             input_shape=(sequence_length, n_features),
             name='LSTM_Layer_1'),
        BatchNormalization(),
        Dropout(0.5),


        LSTM(48,
             return_sequences=False,
             name='LSTM_Layer_2'),
        BatchNormalization(),
        Dropout(0.5),

        Dense(24, activation='relu', name='Dense_Layer'),
        Dropout(0.5),
        Dense(1, activation='sigmoid', name='Output_Layer')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.0005),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


n_features = X_train_seq.shape[2]
model_lstm = build_lstm_model(SEQUENCE_LENGTH, n_features)

print("=" * 65)
model_lstm.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ LSTM_Layer_1 (LSTM)             │ (None, 5, 96)          │        67,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 5, 96)          │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 96)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ LSTM_Layer_2 (LSTM)             │ (None, 48)             │        27,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 48)             │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_Layer (Dense)             │ (None, 24)             │         1,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 24)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 1)              │            25 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 97,585 (381.19 KB)

 Trainable params: 97,297 (380.07 KB)

 Non-trainable params: 288 (1.12 KB)

In [ ]:
#8
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

#Cost-Sensitive Learning
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"Class weights: {class_weight_dict}")

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)



history = model_lstm.fit(
    X_train_seq, y_train,
    validation_data=(X_valid_seq, y_valid),
    epochs=50,
    batch_size=96,
    class_weight=class_weight_dict,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)
model_lstm.save(
    "/content/drive/MyDrive/GP/LstmModels/best_lstm.keras"
)
print("\n done ")

Class weights: {0: np.float64(1.1788194444444444), 1: np.float64(0.8682864450127877)}
Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.6429 - loss: 0.7599 - val_accuracy: 0.5146 - val_loss: 0.6385 - learning_rate: 5.0000e-04
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7541 - loss: 0.5437 - val_accuracy: 0.5146 - val_loss: 0.6074 - learning_rate: 5.0000e-04
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8034 - loss: 0.4668 - val_accuracy: 0.5314 - val_loss: 0.5813 - learning_rate: 5.0000e-04
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8498 - loss: 0.3539 - val_accuracy: 0.6485 - val_loss: 0.5467 - learning_rate: 5.0000e-04
Epoch 5/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8535 - loss: 0.3367 - val_accuracy: 0.7406 - val_loss: 0.5199 - learning_rate: 5.0000e-04
Epoch 6/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.8984 - loss: 0.2901 - val_accuracy: 0.7657 - val_loss: 0.5007 - learning_

In [ ]:
#10
y_pred_proba = model_lstm.predict(X_test_seq, verbose=0).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)

print(" Model results at the sequence level")
print("=" * 60)
print(f"\n   Accuracy:  {acc*100:.2f}%")
print(f"   Precision: {prec:.4f}")
print(f"   Recall:    {rec:.4f}")
print(f"   F1 Score:  {f1:.4f}")

print(classification_report(y_test, y_pred,
                           target_names=['Alert', 'Drowsy'], digits=4))

 Model results at the sequence level

   Accuracy:  71.02%
   Precision: 0.6625
   Recall:    0.6335
   F1 Score:  0.6477
              precision    recall  f1-score   support

       Alert     0.7423    0.7659    0.7539       346
      Drowsy     0.6625    0.6335    0.6477       251

    accuracy                         0.7102       597
   macro avg     0.7024    0.6997    0.7008       597
weighted avg     0.7087    0.7102    0.7092       597




*    TP: Drowsy predicted correctly
*   TN: Alert predicted correctly
*   FP: Wrong Drowsy (Alert predicted as Drowsy)
*   FN: Missed Drowsy (Drowsy predicted as Alert)

In [ ]:
#11
# Sequence Level
# ==============================

from sklearn.metrics import confusion_matrix

print("\nSequence Level - Detailed Metrics")
print("=" * 60)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

TN, FP, FN, TP = cm.ravel()

print("\nConfusion Matrix:")
print("[[TN  FP]")
print(" [FN  TP]]")
print(cm)

# Accuracy
total = TP + TN + FP + FN
accuracy = (TP + TN) / total

print("\nAccuracy:")
print(f"= (TP + TN) / Total")
print(f"= ({TP} + {TN}) / {total}")
print(f"= {(TP + TN)} / {total}")
print(f"= {accuracy:.4f}  ({accuracy*100:.2f}%)")

# Precision
precision = TP / (TP + FP)

print("\nPrecision (Drowsy):")
print(f"= TP / (TP + FP)")
print(f"= {TP} / ({TP} + {FP})")
print(f"= {TP} / {TP + FP}")
print(f"= {precision:.4f}")

# Recall
recall = TP / (TP + FN)

print("\nRecall (Drowsy):")
print(f"= TP / (TP + FN)")
print(f"= {TP} / ({TP} + {FN})")
print(f"= {TP} / {TP + FN}")
print(f"= {recall:.4f}")

# F1 Score
f1 = 2 * (precision * recall) / (precision + recall)

print("\nF1 Score:")
print(f"= 2 × (Precision × Recall) / (Precision + Recall)")
print(f"= 2 × ({precision:.4f} × {recall:.4f}) / ({precision:.4f} + {recall:.4f})")
print(f"= {f1:.4f}")


Sequence Level - Detailed Metrics

Confusion Matrix:
[[TN  FP]
 [FN  TP]]
[[265  81]
 [ 92 159]]

Accuracy:
= (TP + TN) / Total
= (159 + 265) / 597
= 424 / 597
= 0.7102  (71.02%)

Precision (Drowsy):
= TP / (TP + FP)
= 159 / (159 + 81)
= 159 / 240
= 0.6625

Recall (Drowsy):
= TP / (TP + FN)
= 159 / (159 + 92)
= 159 / 251
= 0.6335

F1 Score:
= 2 × (Precision × Recall) / (Precision + Recall)
= 2 × (0.6625 × 0.6335) / (0.6625 + 0.6335)
= 0.6477


In [ ]:
#12
def video_level_lstm(model, X_seq, y_labels, video_ids, threshold=0.5):
    """soft   Voting."""
    probas = model.predict(X_seq, verbose=0).flatten()

    df = pd.DataFrame({
        'video_id':   video_ids,
        'true_label': y_labels,
        'proba':      probas
    })

    results = []
    for video_id, g in df.groupby('video_id'):
        mean_proba = g['proba'].mean()
        final_pred = int(mean_proba >= threshold)
        true_label = int(g['true_label'].iloc[0])

        results.append({
            'video_id':         video_id,
            'subject_id':       video_id.split('_')[0],
            'true_label':       true_label,
            'predicted_label':  final_pred,
            'mean_proba':       round(mean_proba, 3),
            'n_sequences':      len(g)
        })

    return pd.DataFrame(results)


print("Video-level evaluation (Soft Voting) - Thresholds experience")
print("=" * 75)
print(f"\n{'Threshold':<12}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'F1':<10}")
print("-" * 58)

best_t = 0.5
best_f1 = 0
for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]:
    df = video_level_lstm(model_lstm, X_test_seq, y_test, test_video_ids, t)
    a = accuracy_score(df['true_label'], df['predicted_label'])
    p = precision_score(df['true_label'], df['predicted_label'], zero_division=0)
    r = recall_score(df['true_label'], df['predicted_label'], zero_division=0)
    f = f1_score(df['true_label'], df['predicted_label'], zero_division=0)
    print(f"{t:<12}{a:<12.4f}{p:<12.4f}{r:<12.4f}{f:<10.4f}")
    if f > best_f1:
        best_f1 = f
        best_t  = t

print(f"\n best Threshold: {best_t}")

final_df = video_level_lstm(model_lstm, X_test_seq, y_test, test_video_ids, best_t)
final_df['true_class']      = final_df['true_label'].map({0: 'Alert', 1: 'Drowsy'})
final_df['predicted_class'] = final_df['predicted_label'].map({0: 'Alert', 1: 'Drowsy'})
final_df['Status']          = np.where(
    final_df['true_label'] == final_df['predicted_label'], 'true', 'false')

print(f"\n Video details (threshold={best_t}):")
print(final_df[['subject_id', 'true_class', 'predicted_class',
                'mean_proba', 'n_sequences', 'Status']].to_string(index=False))

final_acc = accuracy_score(final_df['true_label'], final_df['predicted_label'])
final_p   = precision_score(final_df['true_label'], final_df['predicted_label'], zero_division=0)
final_r   = recall_score(final_df['true_label'], final_df['predicted_label'], zero_division=0)
final_f1  = f1_score(final_df['true_label'], final_df['predicted_label'], zero_division=0)

print(f"\n Final result at the video level: ")
print(f"   Accuracy:  {final_acc*100:.1f}%")
print(f"   Precision: {final_p:.4f}")
print(f"   Recall:    {final_r:.4f}")
print(f"   F1 Score:  {final_f1:.4f}")


from sklearn.metrics import classification_report

print("\nDetailed classification report (Video Level):")
print(classification_report(
    final_df['true_label'],
    final_df['predicted_label'],
    target_names=['Alert', 'Drowsy'],
    digits=4
))

Video-level evaluation (Soft Voting) - Thresholds experience

Threshold   Accuracy    Precision   Recall      F1        
----------------------------------------------------------
0.2         0.7000      0.6667      0.8000      0.7273    
0.25        0.7000      0.6667      0.8000      0.7273    
0.3         0.7000      0.6667      0.8000      0.7273    
0.35        0.7000      0.6667      0.8000      0.7273    
0.4         0.7000      0.6667      0.8000      0.7273    
0.45        0.7000      0.6667      0.8000      0.7273    
0.5         0.7000      0.6667      0.8000      0.7273    
0.55        0.7000      0.7500      0.6000      0.6667    

 best Threshold: 0.2

 Video details (threshold=0.2):
subject_id true_class predicted_class  mean_proba  n_sequences Status
      A022      Alert          Drowsy       0.506           55  false
      A023      Alert           Alert       0.140           71   true
      A024      Alert          Drowsy       0.708           52  false
      A025   

In [ ]:
#13

# Video Level
# ==============================

from sklearn.metrics import confusion_matrix

print("\nVideo Level - Detailed Metrics")
print("=" * 60)

# Confusion Matrix
cm = confusion_matrix(
    final_df['true_label'],
    final_df['predicted_label']
)

TN, FP, FN, TP = cm.ravel()

print("\nConfusion Matrix:")
print("[[TN  FP]")
print(" [FN  TP]]")
print(cm)

# Accuracy
total = TP + TN + FP + FN
accuracy = (TP + TN) / total

print("\nAccuracy:")
print(f"= (TP + TN) / Total")
print(f"= ({TP} + {TN}) / {total}")
print(f"= {(TP + TN)} / {total}")
print(f"= {accuracy:.4f}  ({accuracy*100:.2f}%)")

# Precision
precision = TP / (TP + FP)

print("\nPrecision (Drowsy):")
print(f"= TP / (TP + FP)")
print(f"= {TP} / ({TP} + {FP})")
print(f"= {TP} / {TP + FP}")
print(f"= {precision:.4f}")

# Recall
recall = TP / (TP + FN)

print("\nRecall (Drowsy):")
print(f"= TP / (TP + FN)")
print(f"= {TP} / ({TP} + {FN})")
print(f"= {TP} / {TP + FN}")
print(f"= {recall:.4f}")

# F1 Score
f1 = 2 * (precision * recall) / (precision + recall)

print("\nF1 Score:")
print(f"= 2 × (Precision × Recall) / (Precision + Recall)")
print(f"= 2 × ({precision:.4f} × {recall:.4f}) / ({precision:.4f} + {recall:.4f})")
print(f"= {f1:.4f}")


Video Level - Detailed Metrics

Confusion Matrix:
[[TN  FP]
 [FN  TP]]
[[3 2]
 [1 4]]

Accuracy:
= (TP + TN) / Total
= (4 + 3) / 10
= 7 / 10
= 0.7000  (70.00%)

Precision (Drowsy):
= TP / (TP + FP)
= 4 / (4 + 2)
= 4 / 6
= 0.6667

Recall (Drowsy):
= TP / (TP + FN)
= 4 / (4 + 1)
= 4 / 5
= 0.8000

F1 Score:
= 2 × (Precision × Recall) / (Precision + Recall)
= 2 × (0.6667 × 0.8000) / (0.6667 + 0.8000)
= 0.7273


In [ ]:
#14
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

BOUNDARY = 0.5

def compute_bsa_bsre(y_true, y_pred, out_scores, boundary=BOUNDARY):

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    out_scores = np.asarray(out_scores).astype(float)

    Cs = (y_pred != y_true).astype(int)

    bsre = np.mean(
        Cs * np.abs(out_scores - boundary) ** 2
    )

    bsa = accuracy_score(y_true, y_pred)

    return bsa, bsre



def compute_vre(y_true, y_pred, out_scores, boundary=0.5):

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    out_scores = np.asarray(out_scores).astype(float)

    Cv = (y_pred != y_true).astype(int)

    video_errors = Cv * np.abs(out_scores - boundary) ** 2

    vre = np.mean(video_errors)

    va = accuracy_score(y_true, y_pred)
    return vre, va

with va-vre and ba-bsre

In [ ]:
#15
def video_level_lstm(model, X_seq, y_labels, video_ids, threshold=0.5):
    """Soft Voting"""

    probas = model.predict(X_seq, verbose=0).flatten()

    df = pd.DataFrame({
        'video_id':   video_ids,
        'true_label': y_labels,
        'proba':      probas
    })

    results = []

    # LOOP THROUGH EVERY VIDEO
    for video_id, g in df.groupby('video_id'):

        mean_proba = g['proba'].mean()
        final_pred = int(mean_proba >= threshold)
        true_label = int(g['true_label'].iloc[0])

        y_pred = (g['proba'] >= threshold).astype(int)

        bsa, bsre = compute_bsa_bsre(
         g['true_label'],
         y_pred,
        g['proba']
      )

        print("\n" + "=" * 70)
        print(f"Video ID: {video_id}")
        print("=" * 70)

        print(f"BSA: {bsa:.4f}")
        print(f"BSRE: {bsre:.6f}")

        # LOOP THROUGH EVERY SEGMENT
        for segment_num, (_, row) in enumerate(g.iterrows(), start=1):

            segment_pred = int(row['proba'] >= threshold)

            print(
                f"Segment {segment_num:<3} | "
                f"Probability: {row['proba']:.4f} | "
                f"Predicted Class: {segment_pred}"
            )

        # SAVE ONE RESULT PER VIDEO
        results.append({
            'video_id':         video_id,
            'subject_id':       video_id.split('_')[0],
            'true_label':       true_label,
            'predicted_label':  final_pred,
            'mean_proba':       round(mean_proba, 3),
            'n_sequences':      len(g)
        })

    return pd.DataFrame(results)


print("Video-level evaluation (Soft Voting) - Thresholds experience")
print("=" * 75)
print(f"\n{'Threshold':<12}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'F1':<10}")
print("-" * 58)

best_t = 0.5
best_f1 = 0

for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]:

    print("\n" + "#" * 80)
    print(f"THRESHOLD = {t}")
    print("#" * 80)

    df = video_level_lstm(
        model_lstm,
        X_test_seq,
        y_test,
        test_video_ids,
        t
    )

    a = accuracy_score(df['true_label'], df['predicted_label'])
    p = precision_score(df['true_label'], df['predicted_label'], zero_division=0)
    r = recall_score(df['true_label'], df['predicted_label'], zero_division=0)
    f = f1_score(df['true_label'], df['predicted_label'], zero_division=0)

    print(f"\nThreshold {t}")
    print(f"Accuracy : {a:.4f}")
    print(f"Precision: {p:.4f}")
    print(f"Recall   : {r:.4f}")
    print(f"F1 Score : {f:.4f}")

    if f > best_f1:
        best_f1 = f
        best_t = t


print(f"\nBest Threshold: {best_t}")

final_df = video_level_lstm(
    model_lstm,
    X_test_seq,
    y_test,
    test_video_ids,
    best_t
)

final_df['true_class'] = final_df['true_label'].map({
    0: 'Alert',
    1: 'Drowsy'
})

final_df['predicted_class'] = final_df['predicted_label'].map({
    0: 'Alert',
    1: 'Drowsy'
})

final_df['Status'] = np.where(
    final_df['true_label'] == final_df['predicted_label'],
    'true',
    'false'
)

print(f"\nVideo details (threshold={best_t}):")

print(
    final_df[
        [
            'subject_id',
            'true_class',
            'predicted_class',
            'mean_proba',
            'n_sequences',
            'Status'
        ]
    ].to_string(index=False)
)

final_acc = accuracy_score(
    final_df['true_label'],
    final_df['predicted_label']
)

final_p = precision_score(
    final_df['true_label'],
    final_df['predicted_label'],
    zero_division=0
)

final_r = recall_score(
    final_df['true_label'],
    final_df['predicted_label'],
    zero_division=0
)

final_f1 = f1_score(
    final_df['true_label'],
    final_df['predicted_label'],
    zero_division=0
)
vre , va = compute_vre(final_df['true_label'],
    final_df['predicted_label'],final_df['mean_proba'])
print(f"\nFinal result at the video level:")
print(f"Accuracy : {final_acc*100:.1f}%")
print(f"Precision: {final_p:.4f}")
print(f"Recall   : {final_r:.4f}")
print(f"F1 Score : {final_f1:.4f}")
print(f"VA  (Video Accuracy): {va:.4f}")
print(f"VRE (Video Ranking Error): {vre:.6f}")

Streaming output truncated to the last 5000 lines.
Segment 47  | Probability: 0.0493 | Predicted Class: 0
Segment 48  | Probability: 0.0426 | Predicted Class: 0
Segment 49  | Probability: 0.0460 | Predicted Class: 0
Segment 50  | Probability: 0.0741 | Predicted Class: 0
Segment 51  | Probability: 0.0595 | Predicted Class: 0
Segment 52  | Probability: 0.0415 | Predicted Class: 0
Segment 53  | Probability: 0.0508 | Predicted Class: 0
Segment 54  | Probability: 0.0487 | Predicted Class: 0
Segment 55  | Probability: 0.0511 | Predicted Class: 0
Segment 56  | Probability: 0.0422 | Predicted Class: 0
Segment 57  | Probability: 0.0372 | Predicted Class: 0
Segment 58  | Probability: 0.0533 | Predicted Class: 0
Segment 59  | Probability: 0.0433 | Predicted Class: 0
Segment 60  | Probability: 0.0531 | Predicted Class: 0
Segment 61  | Probability: 0.0612 | Predicted Class: 0
Segment 62  | Probability: 0.0477 | Predicted Class: 0
Segment 63  | Probability: 0.0511 | Predicted Class: 0
Segment 64  | 

In [ ]:

#  Cross-Validation  LSTM (5-Fold)

from sklearn.model_selection import GroupKFold

print("=" * 70)
print("=" * 70)

train_subjects_seq = np.array([vid.split('_')[0] for vid in train_video_ids])

gkf = GroupKFold(n_splits=5)
fold_train_accs = []
fold_val_accs   = []
fold_f1s        = []

for fold_num, (tr_idx, val_idx) in enumerate(
        gkf.split(X_train_seq, y_train, train_subjects_seq), 1):

    print(f"\n{'─' * 60}")
    print(f" Fold {fold_num}/5")
    print(f"{'─' * 60}")

    X_tr, X_val = X_train_seq[tr_idx], X_train_seq[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    from sklearn.utils.class_weight import compute_class_weight
    if len(np.unique(y_tr)) > 1:
        cw = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
        cw_dict = {0: cw[0], 1: cw[1]}
    else:
        cw_dict = {0: 1.0, 1: 1.0}

    fold_model = Sequential([
        LSTM(64, return_sequences=True,
             input_shape=(SEQUENCE_LENGTH, n_features)),
        BatchNormalization(),
        Dropout(0.3),
        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.3),
        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    fold_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    fold_model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        class_weight=cw_dict,
        callbacks=[EarlyStopping(monitor='val_loss', patience=8,
                                  restore_best_weights=True)],
        verbose=0
    )

    tr_pred = (fold_model.predict(X_tr,  verbose=0).flatten() >= 0.5).astype(int)
    val_pred = (fold_model.predict(X_val, verbose=0).flatten() >= 0.5).astype(int)

    tr_acc  = accuracy_score(y_tr,  tr_pred)
    val_acc = accuracy_score(y_val, val_pred)
    val_f1  = f1_score(y_val, val_pred, zero_division=0)

    fold_train_accs.append(tr_acc)
    fold_val_accs.append(val_acc)
    fold_f1s.append(val_f1)

    print(f"   Train Acc: {tr_acc*100:.2f}%")
    print(f"   Val Acc:   {val_acc*100:.2f}%")
    print(f"   Val F1:    {val_f1:.4f}")
    print(f"   Gap:       {(tr_acc-val_acc)*100:.2f}%")

# ============================================================
print(f"\n{'=' * 70}")
print(f"  Summary of Cross-Validation Results for LSTM: ")
print(f"{'=' * 70}")
print(f"\n   Average Train Accuracy: {np.mean(fold_train_accs)*100:.2f}%")
print(f"   Average Val Accuracy:   {np.mean(fold_val_accs)*100:.2f}% "
      f"(±{np.std(fold_val_accs)*100:.2f}%)")
print(f"   Average F1:             {np.mean(fold_f1s):.4f} "
      f"(±{np.std(fold_f1s):.4f})")
print(f"   Average difference        "
      f"{(np.mean(fold_train_accs)-np.mean(fold_val_accs))*100:.2f}%")



std_val = np.std(fold_val_accs)
print(f"\n Evaluation:")
if std_val < 0.05:
    print("  Excellent! Results are stable (deviation < 5%) — Model is reliable ")
elif std_val < 0.10:
    print("    Good. Average deviation (5-10%) — The model is reasonable ")
else:
    print("    Results are fluctuating (>10%) — Model may be unstable  ")

'\n#  Cross-Validation  LSTM (5-Fold)\n\nfrom sklearn.model_selection import GroupKFold\n\nprint("=" * 70)\nprint("=" * 70)\n\ntrain_subjects_seq = np.array([vid.split(\'_\')[0] for vid in train_video_ids])\n\ngkf = GroupKFold(n_splits=5)\nfold_train_accs = []\nfold_val_accs   = []\nfold_f1s        = []\n\nfor fold_num, (tr_idx, val_idx) in enumerate(\n        gkf.split(X_train_seq, y_train, train_subjects_seq), 1):\n\n    print(f"\n{\'─\' * 60}")\n    print(f" Fold {fold_num}/5")\n    print(f"{\'─\' * 60}")\n\n    X_tr, X_val = X_train_seq[tr_idx], X_train_seq[val_idx]\n    y_tr, y_val = y_train[tr_idx], y_train[val_idx]\n\n    from sklearn.utils.class_weight import compute_class_weight\n    if len(np.unique(y_tr)) > 1:\n        cw = compute_class_weight(\'balanced\', classes=np.unique(y_tr), y=y_tr)\n        cw_dict = {0: cw[0], 1: cw[1]}\n    else:\n        cw_dict = {0: 1.0, 1: 1.0}\n\n    fold_model = Sequential([\n        LSTM(64, return_sequences=True,\n             input_shape=

In [ ]:
# Best threshold from VALIDATION set, then evaluate TEST set

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd

def video_level_lstm_from_probs(probas, y_labels, video_ids, threshold=0.5):
    df = pd.DataFrame({
        "video_id": video_ids,
        "true_label": y_labels,
        "proba": probas
    })

    results = []
    for video_id, g in df.groupby("video_id"):
        mean_proba = g["proba"].mean()
        final_pred = int(mean_proba >= threshold)
        true_label = int(g["true_label"].iloc[0])

        results.append({
            "video_id": video_id,
            "subject_id": video_id.split("_")[0],
            "true_label": true_label,
            "predicted_label": final_pred,
            "mean_proba": mean_proba,
            "n_sequences": len(g)
        })

    return pd.DataFrame(results)


# =========================
# 1) Predict probabilities
# =========================
valid_proba = model_lstm.predict(X_valid_seq, verbose=0).flatten()
test_proba  = model_lstm.predict(X_test_seq, verbose=0).flatten()


# =========================
# 2) Find best threshold on VALIDATION
# =========================
thresholds = [0.45]

best_t = 0.5
best_f1 = -1
results = []

for t in thresholds:
    val_df = video_level_lstm_from_probs(
        valid_proba,
        y_valid,
        valid_video_ids,
        threshold=t
    )

    acc = accuracy_score(val_df["true_label"], val_df["predicted_label"])
    prec = precision_score(val_df["true_label"], val_df["predicted_label"], zero_division=0)
    rec = recall_score(val_df["true_label"], val_df["predicted_label"], zero_division=0)
    f1 = f1_score(val_df["true_label"], val_df["predicted_label"], zero_division=0)

    results.append([t, acc, prec, rec, f1])

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

results_df = pd.DataFrame(
    results,
    columns=["Threshold", "Validation Accuracy", "Validation Precision", "Validation Recall", "Validation F1"]
)

print("Best threshold selected from VALIDATION set")
print("=" * 60)
print(f"Best Threshold: {best_t:.2f}")
print(f"Best Validation F1: {best_f1:.4f}")

display(results_df.sort_values("Validation F1", ascending=False).head(10))


# =========================
# 3) Apply best threshold to TEST
# =========================
final_df = video_level_lstm_from_probs(
    test_proba,
    y_test,
    test_video_ids,
    threshold=best_t
)

test_acc = accuracy_score(final_df["true_label"], final_df["predicted_label"])
test_prec = precision_score(final_df["true_label"], final_df["predicted_label"], zero_division=0)
test_rec = recall_score(final_df["true_label"], final_df["predicted_label"], zero_division=0)
test_f1 = f1_score(final_df["true_label"], final_df["predicted_label"], zero_division=0)

print("\nTEST results using best VALIDATION threshold")
print("=" * 60)
print(f"Threshold: {best_t:.2f}")
print(f"Accuracy:  {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall:    {test_rec:.4f}")
print(f"F1 Score:  {test_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    final_df["true_label"],
    final_df["predicted_label"],
    target_names=["Alert", "Drowsy"],
    digits=4
))

print("\nConfusion Matrix:")
cm = confusion_matrix(final_df["true_label"], final_df["predicted_label"])
print(cm)

final_df["true_class"] = final_df["true_label"].map({0: "Alert", 1: "Drowsy"})
final_df["predicted_class"] = final_df["predicted_label"].map({0: "Alert", 1: "Drowsy"})
final_df["Status"] = np.where(
    final_df["true_label"] == final_df["predicted_label"],
    "Correct",
    "Wrong"
)

display(final_df)

Best threshold selected from VALIDATION set
Best Threshold: 0.45
Best Validation F1: 0.9091


,Threshold,Validation Accuracy,Validation Precision,Validation Recall,Validation F1
0,0.45,0.9,0.833333,1.0,0.909091



TEST results using best VALIDATION threshold
Threshold: 0.45
Accuracy:  0.7000
Precision: 0.6667
Recall:    0.8000
F1 Score:  0.7273

Classification Report:
              precision    recall  f1-score   support

       Alert     0.7500    0.6000    0.6667         5
      Drowsy     0.6667    0.8000    0.7273         5

    accuracy                         0.7000        10
   macro avg     0.7083    0.7000    0.6970        10
weighted avg     0.7083    0.7000    0.6970        10


Confusion Matrix:
[[3 2]
 [1 4]]


,video_id,subject_id,true_label,predicted_label,mean_proba,n_sequences,true_class,predicted_class,Status
0,A022_20260513_190541_frames,A022,0,1,0.506051,55,Alert,Drowsy,Wrong
1,A023_20260513_195602_frames,A023,0,0,0.140401,71,Alert,Alert,Correct
2,A024_20260513_212736_frames,A024,0,1,0.707731,52,Alert,Drowsy,Wrong
3,A025_20260513_233235_frames,A025,0,0,0.083026,91,Alert,Alert,Correct
4,A026_20260514_025227_frames,A026,0,0,0.150031,77,Alert,Alert,Correct
5,D022_20260513_190626_frames,D022,1,1,0.725359,29,Drowsy,Drowsy,Correct
6,D023_20260513_195650_frames,D023,1,1,0.799688,32,Drowsy,Drowsy,Correct
7,D024_20260513_212814_frames,D024,1,1,0.518237,79,Drowsy,Drowsy,Correct
8,D025_20260513_233342_frames,D025,1,0,0.199878,54,Drowsy,Alert,Wrong
9,D026_20260514_025307_frames,D026,1,1,0.616394,57,Drowsy,Drowsy,Correct


In [ ]:
# ============================================================
# LSTM: Window + Stride + Sequence (FINAL CLEAN VERSION)
# - Best hyperparameters from Bayesian search
# - Fixed threshold = 0.25
# - Zero-padding (no videos dropped)
# - Table format matches window-based table (A: / D:)
# - Uses actual valid_df (not random split)
# ============================================================

import gc
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight


# ============================================================
# Settings
# ============================================================

window_sizes    = [5, 8, 10, 15, 20]
strides         = [1, 2, 5]
sequence_lengths = [5, 10, 15]

LSTM_THRESHOLD = 0.3 # fixed threshold

# ============================================================
# Best hyperparameters from Bayesian search
# ============================================================

BATCH_SIZE    = 64
LEARNING_RATE = 0.001
LSTM_1_UNITS  = 64
LSTM_2_UNITS  = 32
DROPOUT_RATE  = 0.3
DENSE_UNITS   = 16
EPOCHS        = 50


# ============================================================
# Helper: add video_id if missing
# ============================================================

def add_video_id_if_missing(df):
    df = df.copy()
    if "video_id" in df.columns:
        return df
    possible_cols = ["video", "video_name", "folder", "file", "filename", "source"]
    for col in possible_cols:
        if col in df.columns:
            df["video_id"] = df[col]
            return df
    df["video_id"] = "video_0"
    return df


# ============================================================
# Build windows (with zero-padding — no videos dropped)
# ============================================================

def build_window_features(df, window_size=8, stride=2):
    df = add_video_id_if_missing(df)
    feature_cols = ["Duration", "Amplitude", "Velocity", "Frequency"]
    rows = []

    for video_id, g in df.groupby("video_id"):
        g = g.sort_values("Blink_ID").reset_index(drop=True)
        label      = g["label"].iloc[0]
        subject_id = g["subject_id"].iloc[0] if "subject_id" in g.columns else None

        # pad if fewer blinks than window_size
        if len(g) < window_size:
            pad_needed = window_size - len(g)
            pad_df = pd.DataFrame([{
                "Blink_ID":   -1,
                "video_id":   video_id,
                "label":      label,
                "subject_id": subject_id,
                "Duration":   0.0,
                "Amplitude":  0.0,
                "Velocity":   0.0,
                "Frequency":  0.0
            }] * pad_needed)
            g = pd.concat([pad_df, g], ignore_index=True)

        for start in range(0, len(g) - window_size + 1, stride):
            window = g.iloc[start:start + window_size]
            rows.append({
                "video_id":   video_id,
                "subject_id": subject_id,
                "label":      label,
                "window_idx": start,
                "data":       window[feature_cols].values
            })

    return pd.DataFrame(rows)


# ============================================================
# Build sequences (with zero-padding — no videos dropped)
# ============================================================

def make_sequences_from_windows(windows_df, sequence_length):
    windows_df = add_video_id_if_missing(windows_df)
    X_seq        = []
    y_seq        = []
    video_ids_seq = []

    for video_id, group in windows_df.groupby("video_id"):
        group = group.reset_index(drop=True)

        # pad if not enough windows to form one sequence
        if len(group) < sequence_length:
            pad_needed = sequence_length - len(group)
            n_features = group["data"].iloc[0].flatten().shape[0]
            pad_row = pd.DataFrame([{
                "video_id":   video_id,
                "label":      group["label"].iloc[0],
                "data":       np.zeros(group["data"].iloc[0].shape, dtype=np.float32),
                "window_idx": -1
            }] * pad_needed)
            group = pd.concat([pad_row, group], ignore_index=True)

        for i in range(0, len(group) - sequence_length + 1):
            seq_df   = group.iloc[i:i + sequence_length]
            seq_data = np.array([
                item.flatten() for item in seq_df["data"].tolist()
            ])
            X_seq.append(seq_data)
            y_seq.append(seq_df["label"].iloc[-1])
            video_ids_seq.append(video_id)

    return (
        np.array(X_seq,         dtype=np.float32),
        np.array(y_seq,         dtype=np.int32),
        np.array(video_ids_seq)
    )


# ============================================================
# Build LSTM model (best hyperparameters)
# ============================================================

def build_lstm_model(sequence_length, n_features):
    model = tf.keras.models.Sequential([
        tf.keras.layers.LSTM(
            LSTM_1_UNITS,
            return_sequences=True,
            input_shape=(sequence_length, n_features)
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(DROPOUT_RATE),

        tf.keras.layers.LSTM(
            LSTM_2_UNITS,
            return_sequences=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(DROPOUT_RATE),

        tf.keras.layers.Dense(DENSE_UNITS, activation="relu"),
        tf.keras.layers.Dropout(DROPOUT_RATE),

        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# ============================================================
# Evaluation — sequence-level + video-level
# ============================================================

def evaluate(model, X_data, y_data, video_ids, threshold):
    probs = model.predict(X_data, verbose=0).flatten()
    preds = (probs >= threshold).astype(int)

    # sequence-level
    seq_acc = accuracy_score(y_data, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_data, preds, labels=[0, 1], zero_division=0
    )

    # video-level (soft voting)
    temp_df = pd.DataFrame({
        "video_id":   video_ids,
        "true_label": y_data,
        "prob":       probs
    })

    video_rows = []
    for vid, g in temp_df.groupby("video_id"):
        mean_prob  = g["prob"].mean()
        video_pred = 1 if mean_prob >= threshold else 0
        video_rows.append({
            "true_label":      g["true_label"].iloc[0],
            "predicted_label": video_pred
        })

    video_df  = pd.DataFrame(video_rows)
    video_acc = accuracy_score(
        video_df["true_label"],
        video_df["predicted_label"]
    )

    return {
        "precision_A": precision[0],
        "precision_D": precision[1],
        "recall_A":    recall[0],
        "recall_D":    recall[1],
        "f1_A":        f1[0],
        "f1_D":        f1[1],
        "seq_acc":     seq_acc,
        "video_acc":   video_acc
    }


def format_row(window, stride, seq_len, r):
    return {
        "Window":       window,
        "Stride":       stride,
        "Sequence":     seq_len,
        "Threshold":    LSTM_THRESHOLD,
        "Precision":    f"A: {r['precision_A']:.2f}\nD: {r['precision_D']:.2f}",
        "Recall":       f"A: {r['recall_A']:.2f}\nD: {r['recall_D']:.2f}",
        "F1-score":     f"A: {r['f1_A']:.2f}\nD: {r['f1_D']:.2f}",
        "Sequence Acc.": f"{r['seq_acc'] * 100:.2f}%",
        "Video Acc.":   f"{r['video_acc'] * 100:.2f}%"
    }


# ============================================================
# Main experiment loop
# ============================================================

all_results = []

for window in window_sizes:
    for stride in strides:
        for seq_len in sequence_lengths:

            print("\n" + "=" * 70)
            print(f"Window={window}  Stride={stride}  Sequence={seq_len}")
            print("=" * 70)

            tf.keras.backend.clear_session()
            gc.collect()

            # build windows
            train_windows = build_window_features(train_df, window, stride)
            valid_windows = build_window_features(valid_df, window, stride)
            test_windows  = build_window_features(test_df,  window, stride)

            # build sequences
            X_train, y_train, train_vids = make_sequences_from_windows(train_windows, seq_len)
            X_valid, y_valid, valid_vids = make_sequences_from_windows(valid_windows, seq_len)
            X_test,  y_test,  test_vids  = make_sequences_from_windows(test_windows,  seq_len)

            print(f"Train: {X_train.shape}  Valid: {X_valid.shape}  Test: {X_test.shape}")

            if len(X_train) == 0 or len(X_valid) == 0 or len(X_test) == 0:
                print("Skipped — empty split.")
                continue

            # class weights
            cw      = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
            cw_dict = {int(c): float(w) for c, w in zip(np.unique(y_train), cw)}

            # build + train model
            n_features = X_train.shape[2]
            model      = build_lstm_model(seq_len, n_features)

            model.fit(
                X_train, y_train,
                validation_data=(X_valid, y_valid),
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                class_weight=cw_dict,
                callbacks=[
                    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=0),
                    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=0)
                ],
                verbose=0
            )

            # evaluate on test set
            results = evaluate(model, X_test, y_test, test_vids, LSTM_THRESHOLD)
            all_results.append(format_row(window, stride, seq_len, results))

            print(f"Sequence Acc: {results['seq_acc']*100:.2f}%  |  Video Acc: {results['video_acc']*100:.2f}%")


# ============================================================
# Final table
# ============================================================

lstm_table = pd.DataFrame(all_results)

print("\nLSTM — Window + Stride + Sequence Results")
display(lstm_table)

# save
lstm_table.to_csv(
    "/content/drive/MyDrive/GP/LstmModels/lstm_sequence_results.csv",
    index=False
)
print("\nSaved to: /content/drive/MyDrive/GP/LstmModels/lstm_sequence_results.csv")


Window=5  Stride=1  Sequence=5
Train: (3638, 5, 20)  Valid: (660, 5, 20)  Test: (1262, 5, 20)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Sequence Acc: 45.40%  |  Video Acc: 50.00%

Window=5  Stride=1  Sequence=10
Train: (3386, 10, 20)  Valid: (610, 10, 20)  Test: (1212, 10, 20)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Sequence Acc: 63.37%  |  Video Acc: 80.00%

Window=5  Stride=1  Sequence=15
Train: (3136, 15, 20)  Valid: (560, 15, 20)  Test: (1162, 15, 20)


...

Sequence Acc: 71.13%  |  Video Acc: 70.00%

Window=20  Stride=5  Sequence=15
Train: (162, 15, 80)  Valid: (14, 15, 80)  Test: (99, 15, 80)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Sequence Acc: 76.77%  |  Video Acc: 70.00%

LSTM — Window + Stride + Sequence Results


,Window,Stride,Sequence,Threshold,Precision,Recall,F1-score,Sequence Acc.,Video Acc.
0,5,1,5,0.3,A: 0.65\nD: 0.43,A: 0.11\nD: 0.92,A: 0.19\nD: 0.59,45.40%,50.00%
1,5,1,10,0.3,A: 0.83\nD: 0.54,A: 0.47\nD: 0.86,A: 0.59\nD: 0.67,63.37%,80.00%
2,5,1,15,0.3,A: 0.76\nD: 0.61,A: 0.68\nD: 0.71,A: 0.72\nD: 0.66,68.93%,70.00%
3,5,2,5,0.3,A: 0.60\nD: 0.42,A: 0.07\nD: 0.93,A: 0.13\nD: 0.58,43.74%,50.00%
4,5,2,10,0.3,A: 0.82\nD: 0.58,A: 0.58\nD: 0.82,A: 0.68\nD: 0.68,68.32%,70.00%
5,5,2,15,0.3,A: 0.76\nD: 0.61,A: 0.70\nD: 0.68,A: 0.73\nD: 0.64,69.32%,70.00%
6,5,5,5,0.3,A: 0.83\nD: 0.60,A: 0.61\nD: 0.83,A: 0.70\nD: 0.69,69.82%,70.00%
7,5,5,10,0.3,A: 0.76\nD: 0.60,A: 0.72\nD: 0.64,A: 0.74\nD: 0.62,69.19%,70.00%
8,5,5,15,0.3,A: 0.78\nD: 0.58,A: 0.76\nD: 0.60,A: 0.77\nD: 0.59,70.73%,70.00%
9,8,1,5,0.3,A: 0.82\nD: 0.49,A: 0.30\nD: 0.91,A: 0.44\nD: 0.63,55.68%,50.00%



Saved to: /content/drive/MyDrive/GP/LstmModels/lstm_sequence_results.csv
